In [1]:
from pathlib import Path


PDF = Path('../data/raw/renters_rights_act_2025.pdf')

## Legal corpus for the Rights agent

Extracts and chunks the Renters' Rights Act 2025 for retrieval-augmented
generation. Chunking follows legislative section boundaries rather than fixed
token counts, so that each retrieved passage is a complete provision.

Each section carries its commencement status. The Act contains provisions that
are prospective, enacted but not yet in force, alongside provisions already
commenced. A retrieval system that returns the correct section but presents an
uncommenced provision as operative would mislead a tenant about their current
rights.

Source: Renters' Rights Act 2025, c. 26, as at 23 June 2026.
Output: `data/processed/act_chunks.json`

In [2]:
import pypdf

reader = pypdf.PdfReader(PDF)
print("Pages:", len(reader.pages))

# Extract all pages into one string
pages = [page.extract_text() for page in reader.pages]
full_text = "\n".join(pages)

print("Characters:", len(full_text))
print()
print(full_text[:1200])

Pages: 275
Characters: 684750

Status: This version of this Act contains provisions that are prospective.
Changes to legislation: There are currently no known outstanding effects 
for the Renters’ Rights Act 2025. (See end of Document for details)
Renters’ Rights Act 2025
2025 CHAPTER 26
An Act to make provision changing the law about rented homes, including provision 
abolishing fixed term assured tenancies and assured shorthold tenancies; imposing 
obligations on landlords and others in relation to rented homes and temporary and 
supported accommodation; and for connected purposes. [27th October 2025]
BE IT ENACTED by the King’s most Excellent Majesty, by and with the advice and consent of 
the Lords Spiritual and Temporal, and Commons, in this present Parliament assembled, and by 
the authority of the same, as follows:—
PART 1
TENANCY REFORM
CHAPTER 1
ASSURED TENANCIES
End of certain kinds of assured tenancy
1 Assured tenancies to be periodic with rent period not exceeding a month
I

In [3]:
import re
# Section headings: a number at line start, then the section title
section_pattern = re.compile(r'^(\d{1,3})\s+([A-Z][^\n]{10,120})$', re.MULTILINE)

matches = list(section_pattern.finditer(full_text))
print("Candidate section headings:", len(matches))
print()
for m in matches[:15]:
    print(f"  s.{m.group(1):>4}  {m.group(2)[:70]}")

Candidate section headings: 595

  s.   1  Assured tenancies to be periodic with rent period not exceeding a mont
  s.   2  Renters’ Rights Act 2025 (c. 26)
  s.   2  Abolition of assured shorthold tenancies
  s.   3  Status: This version of this Act contains provisions that are prospect
  s.   3  Changes to grounds for possession
  s.   4  Renters’ Rights Act 2025 (c. 26)
  s.   5  Status: This version of this Act contains provisions that are prospect
  s.   6  Renters’ Rights Act 2025 (c. 26)
  s.   4  Possession for anti-social behaviour: relevant factors
  s.   7  Status: This version of this Act contains provisions that are prospect
  s.   5  Form of notice of proceedings for possession
  s.   6  Statutory procedure for increases of rent
  s.   8  Renters’ Rights Act 2025 (c. 26)
  s.   9  Status: This version of this Act contains provisions that are prospect
  s.   7  Challenging amount or increase of rent


In [4]:
# Page headers and status banners repeat on every page; exclude them
noise = ('Renters', 'Status:', 'Changes to legislation', 'Document Generated')

sections = [
    m for m in matches
    if not m.group(2).startswith(noise)
]

print("Sections after filtering:", len(sections))
print()
for m in sections[:20]:
    print(f"  s.{m.group(1):>4}  {m.group(2)[:70]}")

Sections after filtering: 321

  s.   1  Assured tenancies to be periodic with rent period not exceeding a mont
  s.   2  Abolition of assured shorthold tenancies
  s.   3  Changes to grounds for possession
  s.   4  Possession for anti-social behaviour: relevant factors
  s.   5  Form of notice of proceedings for possession
  s.   6  Statutory procedure for increases of rent
  s.   7  Challenging amount or increase of rent
  s.   8  Prohibition of rent in advance after lease entered into (except initia
  s.   9  Prohibition of rent in advance before lease entered into
  s.  10  Repayment of rent paid for days after end of tenancy
  s.  11  Right to request permission to keep a pet
  s.  12  Duty of landlord and contractor to give statement of terms etc
  s.  13  Other duties
  s.  14  Landlords acting through others
  s.  15  Landlords etc: financial penalties and offences
  s.  16  Financial penalties: procedure, appeals and enforcement
  s.   1  Before imposing a financial penalty o

In [5]:
# Headings may be as short as "Extent"; allow shorter titles
section_pattern = re.compile(r'^(\d{1,3})\s+([A-Z][^\n]{4,120})$', re.MULTILINE)
matches = list(section_pattern.finditer(full_text))

noise = ('Renters', 'Status:', 'Changes to legislation', 'Document Generated')
sentence_starts = ('A ', 'An ', 'If ', 'Before ', 'After ', 'Where ',
                   'In ', 'This ', 'For ', 'It ', 'On ', 'When ',
                   'Subsection', 'Section', 'But ', 'Nothing ')

sections = [
    m for m in matches
    if not m.group(2).startswith(noise)
    and not m.group(2).startswith(sentence_starts)
    and not m.group(2).rstrip().endswith(('—', '.', ':', ';', ','))
]

nums = [int(m.group(1)) for m in sections]
print("Sections:", len(sections))
print("Missing:", sorted(set(range(1, 150)) - set(nums)))
print("Duplicates:", [n for n in set(nums) if nums.count(n) > 1])

Sections: 164
Missing: [83]
Duplicates: [1, 2, 6, 7, 8, 9, 11, 12, 13]


In [6]:
# Find where sections 18 and 41 should be
for n in ['18', '41']:
    idx = full_text.find(f'\n{n} ')
    while idx != -1:
        snippet = full_text[idx:idx+120].replace('\n', ' | ')
        if not snippet.startswith(f'\n{n} Renters'):
            print(f"s.{n}: {snippet[:110]}")
            break
        idx = full_text.find(f'\n{n} ', idx + 1)
    print()

s.18:  | 18 Renters’ Rights Act 2025 (c. 26) | Part 1 – Tenancy reform | Chapter 1 – Assured tenancies | Document Ge

s.41:  | 41 No prohibition on taking income into account | Nothing in this Chapter prohibits taking a person’s incom



In [7]:
print("Total sections:", len(sections))
print("Range:", min(nums), "to", max(nums))
print("Missing:", sorted(set(range(1, max(nums)+1)) - set(nums)))

Total sections: 164
Range: 1 to 149
Missing: [83]


In [8]:
for n in [18, 67, 75, 76, 83, 92, 101, 131, 144]:
    for m in re.finditer(rf'\n{n} (\S[^\n]{{5,100}})', full_text):
        t = m.group(1)
        if not t.startswith(('Renters', 'Status', 'Changes', 'Document')):
            print(f"s.{n}: {t[:85]}")
            break

s.18: No criminal liability of the Crown under Part 1 of 1988 Act
s.67: Offences
s.75: The database
s.76: The database operator
s.83: Entries relating to banning orders, offences, financial penalties, etc.
s.92: Offences
s.101: The standard of MOD accommodation
s.131: Offences
s.144: Extent


In [9]:
# Real sections appear in ascending order. Anything numbered lower than a
# section already seen is inserted text carrying its own numbering.
clean = []
highest = 0
for m in sections:
    n = int(m.group(1))
    if n > highest:
        clean.append(m)
        highest = n

nums = [int(m.group(1)) for m in clean]
print("Sections:", len(clean))
print("Missing:", sorted(set(range(1, 150)) - set(nums)))
print("Duplicates:", [n for n in set(nums) if nums.count(n) > 1])

Sections: 148
Missing: [83]
Duplicates: []


In [10]:
# A heading may legitimately end in "etc." — exclude that from the
# punctuation filter used to reject sentence fragments
sections = [
    m for m in matches
    if not m.group(2).startswith(noise)
    and not m.group(2).startswith(sentence_starts)
    and not (m.group(2).rstrip().endswith(('—', '.', ':', ';', ','))
             and not m.group(2).rstrip().endswith('etc.'))
]

clean = []
highest = 0
for m in sections:
    n = int(m.group(1))
    if n > highest:
        clean.append(m)
        highest = n

nums = [int(m.group(1)) for m in clean]
print("Sections:", len(clean))
print("Missing:", sorted(set(range(1, 150)) - set(nums)))
print("Duplicates:", [n for n in set(nums) if nums.count(n) > 1])

Sections: 149
Missing: []
Duplicates: []


In [11]:
chunks = []

for i, m in enumerate(clean):
    start = m.start()
    end = clean[i + 1].start() if i + 1 < len(clean) else len(full_text)
    body = full_text[start:end]

    # Remove repeating page furniture from within the section text
    body = re.sub(r'\n\d+\s+Renters[^\n]*', '', body)
    body = re.sub(r'\nStatus: This version[^\n]*', '', body)
    body = re.sub(r'\nChanges to legislation:[^\n]*', '', body)
    body = re.sub(r'\nDocument Generated[^\n]*', '', body)
    body = re.sub(r'\n{3,}', '\n\n', body).strip()

    chunks.append({
        'section': int(m.group(1)),
        'heading': m.group(2).strip(),
        'text': body,
        'chars': len(body),
    })

In [12]:
import pandas as pd
summary = pd.DataFrame(chunks)
print("Chunks:", len(summary))
print(summary['chars'].describe().round(0).to_string())
print()
print(summary[['section', 'heading', 'chars']].head(10).to_string(index=False))

Chunks: 149
count       149.0
mean       4225.0
std       18021.0
min         214.0
25%        1001.0
50%        1931.0
75%        3589.0
max      220150.0

 section                                                                       heading  chars
       1       Assured tenancies to be periodic with rent period not exceeding a month   2340
       2                                      Abolition of assured shorthold tenancies    554
       3                                             Changes to grounds for possession   7712
       4                        Possession for anti-social behaviour: relevant factors   1255
       5                                  Form of notice of proceedings for possession    537
       6                                     Statutory procedure for increases of rent   6207
       7                                        Challenging amount or increase of rent   6657
       8 Prohibition of rent in advance after lease entered into (except initial rent)   48

In [13]:
print(summary.nlargest(5, 'chars')[['section', 'heading', 'chars']].to_string(index=False))

 section                                                                  heading  chars
     149                                                              Short title 220150
      43   Discrimination relating to children or benefits status: Welsh language  15794
      44 Discrimination relating to children or benefits status: English language  15131
      50                   Discrimination relating to children or benefits status  14727
      15                          Landlords etc: financial penalties and offences  10631


In [14]:
# Everything after the final section heading includes the schedules, which
# amend other legislation rather than stating provisions a tenant would ask
# about. The corpus is cut at the start of the schedules.
last = chunks[-1]
cut = re.search(r'\nSCHEDULE[S]?\s*\n', last['text'])

if cut:
    last['text'] = last['text'][:cut.start()].strip()
    last['chars'] = len(last['text'])
    print("s.149 truncated to", last['chars'], "characters")
else:
    print("Schedule marker not found; inspect the text")
    print(last['text'][:600])

summary = pd.DataFrame(chunks)
print()
print(summary['chars'].describe().round(0).to_string())

s.149 truncated to 417 characters

count      149.0
mean      2750.0
std       2764.0
min        214.0
25%        995.0
50%       1878.0
75%       3501.0
max      15794.0


In [15]:
# Each section carries a commencement note recording whether it is in force
for c in chunks:
    note = re.search(
        r'Commencement Information(.*?)(?=\n[A-Z]{2,}|\Z)',
        c['text'], re.DOTALL
    )
    if note:
        n = ' '.join(note.group(1).split())
        c['commencement'] = n[:400]
        c['in_force'] = 'in force at' in n.lower()
    else:
        c['commencement'] = None
        c['in_force'] = None

In [16]:
summary = pd.DataFrame(chunks)
print("Sections with a commencement note:", summary['commencement'].notna().sum())
print("Recorded as in force:", (summary['in_force'] == True).sum())
print("Not in force:", (summary['in_force'] == False).sum())
print()
for c in chunks[:3]:
    print(f"s.{c['section']}: {c['commencement'][:150] if c['commencement'] else 'none'}")
    print()

Sections with a commencement note: 149
Recorded as in force: 149
Not in force: 0

s.1: I1 S. 1 not in force at Royal Assent, see s. 145(1)(8) I2 S. 1 in force at 1.5.2026 for specified purposes by S.I. 2026/421, reg. 2(a)

s.2: I3 S. 2 not in force at Royal Assent, see s. 145(1)(8) I4 S. 2 in force at 1.5.2026 for specified purposes by S.I. 2026/421, reg. 2(a) Grounds for pos

s.3: I5 S. 3 not in force at Royal Assent, see s. 145(1)(8) I6 S. 3 in force at 1.5.2026 for specified purposes by S.I. 2026/421, reg. 2(a) (with reg. 6)



In [17]:
from datetime import date, datetime

TODAY = date.today()

for c in chunks:
    n = c['commencement'] or ''

    # Commencement dates appear as d.m.yyyy
    raw = re.findall(r'in force at (\d{1,2})\.(\d{1,2})\.(\d{4})', n)
    dates = [date(int(y), int(m), int(d)) for d, m, y in raw]

    c['commencement_dates'] = [d.isoformat() for d in dates]
    c['has_commencement_date'] = len(dates) > 0

    # A provision is in force now only if a commencement date has passed
    c['in_force_now'] = any(d <= TODAY for d in dates)

    # Dates still in the future
    future = [d for d in dates if d > TODAY]
    c['pending_dates'] = [d.isoformat() for d in future]

    c['partial'] = 'for specified purposes' in n

In [18]:
summary = pd.DataFrame(chunks)
print("Assessed as at:", TODAY.isoformat())
print("In force now:  ", summary['in_force_now'].sum())
print("Not yet in force:", (~summary['in_force_now']).sum())
print("With future commencement dates:",
      summary['pending_dates'].apply(len).gt(0).sum())

Assessed as at: 2026-11-26
In force now:   98
Not yet in force: 51
With future commencement dates: 0


In [19]:
def jurisdiction(section):
    """
    Sections 33-42 are Chapter 3 (England), 43-49 Chapter 4 (Wales),
    50-55 Chapter 5 (Scotland). These contain parallel provisions.
    """
    if 33 <= section <= 42:
        return 'England'
    if 43 <= section <= 49:
        return 'Wales'
    if 50 <= section <= 55:
        return 'Scotland'
    return 'UK'


for c in chunks:
    c['jurisdiction'] = jurisdiction(c['section'])

print("done")

done


In [20]:
import json
from pathlib import Path

out = Path('../data/processed/act_chunks.json')

with open(out, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

summary = pd.DataFrame(chunks)

print("Saved", len(chunks), "chunks to", out)
print("Assessed as at:", TODAY.isoformat())
print()
print("In force now:", summary['in_force_now'].sum())
print("Not yet commenced:", (~summary['in_force_now']).sum())
print("Partial commencement:", summary['partial'].sum())

Saved 149 chunks to ..\data\processed\act_chunks.json
Assessed as at: 2026-11-26

In force now: 98
Not yet commenced: 51
Partial commencement: 64


### Embedding the corpus

Each section is converted into a 384-dimensional vector using the
all-MiniLM-L6-v2 sentence-transformer model. Sections with similar meaning
produce vectors that sit close together, allowing retrieval by meaning rather
than by keyword match.

In [21]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding dimensions:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimensions: 384


C:\Users\azmat\AppData\Local\Temp\ipykernel_12480\4013596461.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", model.get_sentence_embedding_dimension())


In [22]:
# Heading and text combined, so the section title contributes to the match
texts = [f"Section {c['section']}: {c['heading']}\n\n{c['text']}" for c in chunks]

embeddings = model.encode(texts, show_progress_bar=True)
print("Embeddings:", embeddings.shape)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embeddings: (149, 384)


In [23]:
import chromadb

client = chromadb.PersistentClient(path='../data/processed/chroma')

try:
    client.delete_collection('renters_rights_act')
except Exception:
    pass

collection = client.create_collection('renters_rights_act')

texts = [f"Section {c['section']}: {c['heading']}\n\n{c['text']}" for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

collection.add(
    ids=[f"s{c['section']}" for c in chunks],
    documents=texts,
    embeddings=embeddings.tolist(),
            metadatas=[{
        'section': c['section'],
        'heading': c['heading'],
        'in_force_now': bool(c['in_force_now']),
        'partial': bool(c['partial']),
        'jurisdiction': c['jurisdiction'],
        'commencement': c['commencement'] or '',
        'assessed_at': TODAY.isoformat(),
    } for c in chunks],
)

print("Stored:", collection.count(), "sections, assessed as at", TODAY.isoformat())

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Stored: 149 sections, assessed as at 2026-11-26


In [24]:
def search(question, k=3):
    """Retrieve the k most similar sections to a question."""
    q_emb = model.encode([question])
    res = collection.query(query_embeddings=q_emb.tolist(), n_results=k)

    for doc, meta, dist in zip(res['documents'][0],
                               res['metadatas'][0],
                               res['distances'][0]):
        status = 'in force' if meta['in_force_now'] else 'NOT YET COMMENCED'
        partial = ' (partial)' if meta['partial'] else ''
        print(f"s.{meta['section']}  distance={dist:.3f}  [{status}{partial}]")
        print(f"   {meta['heading']}")
        print()


search("Can my landlord refuse to rent to me because I have children?")

s.33  distance=0.960  [in force (partial)]
   Discrimination relating to children

s.44  distance=1.090  [in force]
   Discrimination relating to children or benefits status: English language

s.50  distance=1.106  [in force]
   Discrimination relating to children or benefits status



In [25]:
print("=" * 60)
search("How much notice must a landlord give to end a tenancy?")
print("=" * 60)
search("What is the recipe for carbonara?")
print("=" * 60)
search("Can I keep a dog in my rented flat?")

s.20  distance=0.815  [in force (partial)]
   Notices to quit by tenants under assured tenancies: timing

s.13  distance=0.830  [in force (partial)]
   Other duties

s.21  distance=0.857  [in force (partial)]
   Notices to quit by tenants under assured tenancies: other

s.141  distance=1.824  [NOT YET COMMENCED]
   Power of Welsh Ministers to make consequential provision

s.143  distance=1.856  [NOT YET COMMENCED]
   Power of Secretary of State to make consequential provision

s.140  distance=1.877  [NOT YET COMMENCED]
   Regulations

s.11  distance=0.955  [in force (partial)]
   Right to request permission to keep a pet

s.60  distance=1.284  [NOT YET COMMENCED]
   Remedying of hazards occurring in dwelling-houses in England

s.33  distance=1.293  [in force (partial)]
   Discrimination relating to children

